# Notebook 03 — LangGraph Basics

## LangGraph คืออะไร?

**LangGraph** เป็น library สำหรับสร้าง **stateful AI agents** ในรูปแบบ **graph**

แทนที่จะเขียน if/else ยาวๆ เพื่อ route AI workflow, LangGraph ให้เราประกาศ:
- **Nodes** = แต่ละ step ที่ทำงาน (function)
- **Edges** = เส้นทางระหว่าง nodes (routing logic)
- **State** = ข้อมูลที่ส่งผ่าน graph

---

## 3 ส่วนหลักของ LangGraph

```
┌─────────────────────────────────────────────────────────┐
│                       STATE                             │
│  TypedDict ที่เก็บข้อมูลทั้งหมดระหว่าง execution      │
│  เช่น: messages, task_status, user_goal, answer         │
└─────────────────────────────────────────────────────────┘
         ↓ ส่งผ่าน nodes
┌─────────────────────────────────────────────────────────┐
│                       NODES                             │
│  async functions ที่รับ State และ return State ใหม่     │
│  เช่น: session_gate, intent_classify, build_answer      │
└─────────────────────────────────────────────────────────┘
         ↓ connected by
┌─────────────────────────────────────────────────────────┐
│                       EDGES                             │
│  กำหนดว่า node ไหนรันต่อ                               │
│  Normal: A → B (เสมอ)                                  │
│  Conditional: A → B หรือ A → C (ขึ้นกับ state)         │
└─────────────────────────────────────────────────────────┘
```


## 1. State — TypedDict

State คือ "กล่อง" ที่เก็บข้อมูลทั้งหมดระหว่าง graph execution
ใช้ `TypedDict` ประกาศ fields

In [ ]:
from typing import TypedDict, List, Optional, Literal
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

# จำลอง AgentStateStub จาก src/agent/shared/state_stub.py

# TypeAlias
ConversationMode = Literal["AI", "HANDOFF"]
TaskStatus = Literal["ACTIVE", "COMPLETED", "STALLED"]
RouteModule = Literal[
    "hospital_info_agent_module",
    "nurse_agent_module", 
    "package_agent_module",
    "concierge_agent_module"
]

class AgentStateStub(TypedDict, total=False):
    # ─── Message History ───
    messages: List[BaseMessage]         # ประวัติการสนทนา
    latest_user_input: str              # ข้อความล่าสุดจากผู้ใช้
    last_user_message: str              # หลังผ่าน session_gate
    
    # ─── LINE Integration ───
    line_user_id: str                   # LINE User ID
    line_account: str                   # "dev" หรือ "official"
    line_channel_access_token: str      # Token สำหรับ LINE API
    postback_data: str                  # Data จาก postback event
    
    # ─── Conversation Control ───
    conversation_mode: ConversationMode # AI หรือ HANDOFF
    task_status: TaskStatus             # ACTIVE, COMPLETED, STALLED
    
    # ─── Task Tracking ───
    active_task_id: str                 # UUID ของ task ปัจจุบัน
    user_goal: str                      # เป้าหมายของผู้ใช้ (จาก LLM)
    last_user_message: str              # ข้อความล่าสุด
    
    # ─── Routing ───
    route_module: RouteModule           # module ที่จะ route ไป
    current_agent_module: str           # module ที่กำลัง execute
    
    # ─── Hospital Info Module ───
    topic_intents: List[str]            # หัวข้อที่สนใจ
    hospitals: List[str]                # hospital IDs ที่เลือก
    hospital_results: List[dict]        # ข้อมูลจาก CSV
    ready_to_answer: bool
    need_ask_more_info: bool
    
    # ─── Output ───
    final_answer: str                   # คำตอบสุดท้าย (text)
    line_messages: List[dict]           # Flex/complex messages
    quick_replies: List[dict]           # Quick reply buttons


# ตัวอย่าง state เริ่มต้น
initial_state: AgentStateStub = {
    "messages": [],
    "line_user_id": "U1234567890",
    "line_account": "dev",
    "conversation_mode": "AI",
    "task_status": "COMPLETED",
    "latest_user_input": "อยากรู้ข้อมูลโรงพยาบาล",
}

print("Initial State:")
for k, v in initial_state.items():
    print(f"  {k}: {v!r}")

## 2. Nodes — Functions ที่รับและ return State

In [ ]:
from typing import Any

# Node = async function ที่รับ state dict และ return dict (update state)

async def session_gate_node(state: AgentStateStub) -> dict:
    """
    Node แรก: เตรียม session
    - ดึง last_user_message จาก messages หรือ latest_user_input
    - ไม่เปลี่ยน routing logic ใน node นี้
    """
    # ดึงข้อความล่าสุด
    last_user_message = state.get("latest_user_input") or ""
    if not last_user_message:
        # fallback: ดูจาก messages history
        for msg in reversed(state.get("messages", [])):
            if hasattr(msg, 'type') and msg.type == 'human':
                last_user_message = msg.content
                break
    
    print(f"[session_gate] last_user_message = '{last_user_message}'")
    
    # Return เฉพาะ fields ที่เปลี่ยน
    return {"last_user_message": last_user_message}


async def intent_classify_node(state: AgentStateStub) -> dict:
    """
    Node: วิเคราะห์ intent ของผู้ใช้
    - เรียก LLM เพื่อสรุป user_goal
    - เรียก LLM อีกครั้งเพื่อ route ไปยัง module
    """
    user_msg = state.get("last_user_message", "")
    print(f"[intent_classify] Analyzing: '{user_msg}'")
    
    # Mock: ในของจริงเรียก LLM
    # user_goal = await _call_user_goal(user_msg)
    # route = await _call_route(user_goal)
    
    user_goal = f"ผู้ใช้ต้องการ: {user_msg}"  # mock
    route = "hospital_info_agent_module"        # mock routing decision
    
    print(f"[intent_classify] user_goal = '{user_goal}'")
    print(f"[intent_classify] route = '{route}'")
    
    return {
        "user_goal": user_goal,
        "route_module": route,
        "task_status": "ACTIVE",
    }


async def hospital_info_node(state: AgentStateStub) -> dict:
    """
    Module Node: ตอบคำถามเกี่ยวกับโรงพยาบาล
    (ในของจริงเป็น subgraph)
    """
    goal = state.get("user_goal", "")
    print(f"[hospital_info] Processing goal: '{goal}'")
    
    # Mock answer
    answer = f"ข้อมูลโรงพยาบาลสำหรับ: {goal}"
    
    return {
        "final_answer": answer,
        "task_status": "COMPLETED",
    }


# ทดสอบ nodes แบบ manual
import asyncio

async def test_nodes():
    state = dict(initial_state)
    print("=== Testing Nodes ===")
    print()
    
    # Node 1
    update1 = await session_gate_node(state)
    state.update(update1)
    print()
    
    # Node 2
    update2 = await intent_classify_node(state)
    state.update(update2)
    print()
    
    # Node 3
    update3 = await hospital_info_node(state)
    state.update(update3)
    print()
    
    print("=== Final State ===")
    for k in ['last_user_message', 'user_goal', 'route_module', 'task_status', 'final_answer']:
        print(f"  {k}: {state.get(k)!r}")

asyncio.run(test_nodes())

## 3. Edges — Normal vs Conditional

In [ ]:
# ─── Normal Edge ───
# A → B เสมอ ไม่มีเงื่อนไข
#
# graph.add_edge("node_a", "node_b")
#
# ─── Conditional Edge ───  
# A → B หรือ C ขึ้นกับ function ที่ return string

def router_after_session_gate(state: AgentStateStub) -> str:
    """
    Router function: return ชื่อ node ถัดไป
    โค้ดจริง: src/agent/stub_nodes/session_gate_stub.py
    """
    conversation_mode = state.get("conversation_mode", "AI")
    task_status = state.get("task_status", "COMPLETED")
    
    if conversation_mode == "HANDOFF":
        print("Route → handoff_mode (user ถูก handoff ไปที่ agent)")  
        return "handoff_mode"
    
    if task_status == "ACTIVE":
        print("Route → resume_router (task กำลังดำเนินอยู่)")  
        return "resume_router"
    
    print("Route → intent_classify (task ใหม่)")  
    return "intent_classify"


# ทดสอบ routing logic
print("Test Case 1: mode=AI, status=COMPLETED (new task)")
r1 = router_after_session_gate({"conversation_mode": "AI", "task_status": "COMPLETED"})
print(f"  Next node: {r1}")
print()

print("Test Case 2: mode=AI, status=ACTIVE (ongoing task)")
r2 = router_after_session_gate({"conversation_mode": "AI", "task_status": "ACTIVE"})
print(f"  Next node: {r2}")
print()

print("Test Case 3: mode=HANDOFF (handoff to human)")
r3 = router_after_session_gate({"conversation_mode": "HANDOFF", "task_status": "COMPLETED"})
print(f"  Next node: {r3}")

## 4. สร้าง Graph จริงๆ

ใช้ `StateGraph` จาก `langgraph`

In [ ]:
# ติดตั้งก่อน: pip install langgraph
# from langgraph.graph import StateGraph, START, END

# จำลองการสร้าง graph (ไม่รัน จริงเพราะต้องใช้ dependencies)

graph_building_code = '''
from langgraph.graph import StateGraph, START, END

# 1. สร้าง graph พร้อมกำหนด State type
builder = StateGraph(AgentStateStub)

# 2. เพิ่ม Nodes
builder.add_node("session_gate",       session_gate_node)
builder.add_node("intent_classify",    intent_classify_node)
builder.add_node("resume_router",      resume_router_node)
builder.add_node("handoff_mode",       handoff_mode_node)
builder.add_node("hospital_info",      hospital_info_node)
builder.add_node("nurse",              nurse_node)
builder.add_node("package",            package_node)
builder.add_node("concierge",          concierge_node)

# 3. เพิ่ม Edges
builder.add_edge(START, "session_gate")  # เริ่มที่ session_gate

# Conditional edge จาก session_gate
builder.add_conditional_edges(
    "session_gate",
    router_after_session_gate,  # router function
    {
        "intent_classify": "intent_classify",
        "resume_router":   "resume_router",
        "handoff_mode":    "handoff_mode",
    }
)

# Conditional edge จาก intent_classify
builder.add_conditional_edges(
    "intent_classify",
    router_after_intent_classify,
    {
        "hospital_info": "hospital_info",
        "nurse":         "nurse",
        "package":       "package",
        "concierge":     "concierge",
    }
)

# Module nodes ทั้งหมดไปที่ END
builder.add_edge("hospital_info", END)
builder.add_edge("nurse",         END)
builder.add_edge("package",       END)
builder.add_edge("concierge",     END)
builder.add_edge("handoff_mode",  END)

# 4. Compile graph
app = builder.compile()
'''

print(graph_building_code)
print("\nหมายเหตุ: โค้ดจริงอยู่ที่ src/agent/graph_stub.py")

## 5. Invoke Graph

หลัง compile แล้ว เรียกใช้ด้วย `ainvoke()`

In [ ]:
# วิธี invoke graph (โค้ดจริงใน ai_runtime.py)

invoke_code = '''
# config สำหรับ thread isolation (แต่ละ user มี thread ของตัวเอง)
config = {
    "configurable": {
        "thread_id": user_id  # ใช้ LINE user_id เป็น thread ID
    }
}

# invoke แบบ async
state_out = await app.ainvoke(state_in, config=config)

# ดึงผลลัพธ์
final_answer = state_out.get("final_answer")
line_messages = state_out.get("line_messages", [])
quick_replies = state_out.get("quick_replies", [])
'''
print(invoke_code)

# ทำไมต้องใช้ thread_id?
print("\nทำไมต้องใช้ thread_id?")
print("-" * 50)
print("""
LangGraph สามารถ 'checkpointing' state ลง persistent storage
thread_id ทำให้แต่ละ user มี state แยกจากกัน

User A (thread_id=UA123): state_A
User B (thread_id=UB456): state_B  ← แยก memory กัน

ในโปรเจคนี้: session store ทำ stateful ง่ายกว่า
โดยส่ง state เข้า/ออก graph ทุก request
""")

## 6. Subgraph — Graph ซ้อน Graph

ใน LangGraph เราสามารถสร้าง **subgraph** แล้ว embed เป็น node ใน main graph ได้

In [ ]:
# โปรเจคนี้ใช้ subgraph สำหรับแต่ละ module
# เช่น hospital_info_agent_module เป็น subgraph ของตัวเอง

subgraph_example = '''
# ─── Hospital Info Module (subgraph) ───
# src/agent/stub_nodes/hospital_info_agent_module/subgraph.py

hospital_builder = StateGraph(AgentStateStub)

# Nodes ใน subgraph
hospital_builder.add_node("route_hospital_info",   route_node)
hospital_builder.add_node("extract_info_tool",     extract_info_node)
hospital_builder.add_node("ask_more_info_tool",    ask_more_info_node)
hospital_builder.add_node("fetch_data_tool",       fetch_data_node)
hospital_builder.add_node("build_answer_tool",     build_answer_node)
hospital_builder.add_node("oos_check_tool",        oos_check_node)
hospital_builder.add_node("handoff_postback_tool", handoff_postback_node)

hospital_builder.add_edge(START, "route_hospital_info")
hospital_builder.add_conditional_edges(
    "route_hospital_info",
    router_hospital_info,  # เลือก tool ถัดไป
    { ... }
)
# build_answer_tool และ oos_check_tool → END

hospital_subgraph = hospital_builder.compile()

# ─── Main Graph ใช้ subgraph เป็น node ───
main_builder.add_node("hospital_info", hospital_subgraph)  # ← embed!
'''
print(subgraph_example)

print("\nประโยชน์ของ subgraph:")
print("  • แยก logic ของแต่ละ module ออกจากกัน")
print("  • ทดสอบแต่ละ module แยกได้")
print("  • Main graph สะอาด ไม่ซับซ้อน")

## สรุป LangGraph Concepts

```
LangGraph = Directed Graph สำหรับ AI Workflows

┌─────────┐    ┌─────────┐    ┌─────────────┐
│  STATE  │    │  NODES  │    │    EDGES    │
│TypedDict│───►│functions│───►│ routing     │
│ เก็บ   │    │ แต่ละ   │    │ logic       │
│ข้อมูล  │    │ step    │    │(conditional)│
└─────────┘    └─────────┘    └─────────────┘

การทำงาน:
1. invoke(initial_state)
2. START → node_1 → node_2 → ... → END
3. แต่ละ node return dict (update state)
4. Router function เลือก node ถัดไป
5. จบที่ END → return final_state
```

**Notebook ถัดไป:** `04_agent_graph.ipynb` — ดู graph จริงของโปรเจค